# 🤘 Cache Me If You Can — Act I: the getaway

To answer a request, a language model does two things:

- **Prefill:** it reads the whole prompt, already split into tokens, in one pass, and builds its internal state (the KV cache, which you build yourself in Act II).
- **Decode:** it then writes the answer one token at a time. Each new token is added to that state, and every next token needs another full pass through the model.

Prefill decides the time to first token (TTFT). Decode decides how many tokens per second arrive afterwards. In this act you measure both on a real model and work out a speed limit for decode: how fast it could go if reading the weights from memory were the only cost.

**The model:** DeepSeek-R1-Distill-Qwen-1.5B, with 1.5 billion parameters in 28 layers, run the way you would run it in production (Act III says more about the software behind it). Most of its weights are stored as `bfloat8_b` (about 1 byte per number), some as `bfloat16` (2 bytes per number); Act III explains both formats.

**Why two models today.** This production setup is thousands of lines of code. Acts II and IV switch to GPT-2 small, written in about a hundred lines of TT-NN, so you can read and change every line of it.

**Trace.** When the model runs, each operation (for example `ttnn.matmul`) is sent to the card as a command. A full model has hundreds or thousands of them, and the same commands repeat at every decode step. To avoid sending them again and again, TT-NN can record them once and replay the recording at every later step, so the host computer does not resend them. This is called **trace**, and it is switched on throughout this tutorial.

In [ ]:
import sys
from pathlib import Path
REPO = Path.cwd().resolve()
while REPO != REPO.parent and not (REPO / "2026" / "common").is_dir():
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "2026"))
from common import config, wait_for_warmup
wait_for_warmup()
import json, os
import matplotlib.pyplot as plt
from common.llm import LLM
llm = LLM(max_batch_size=1)
print(f"model ready in {llm.load_s:.0f} s on an {llm.card['name']} "
      f"({llm.card['dram_gb']} GB, {llm.card['bandwidth_gbs']} GB/s)")

## 1. It talks

One prompt, one answer, printed token by token as it is generated. This is a *reasoning* model: it first thinks out loud, and only then answers. With 160 tokens it usually stops while still thinking, which is fine: here we only need its speed.

In [ ]:
answer_tokens, shown = [], [""]
def show(step, tokens):                     # print the answer as it is generated
    answer_tokens.append(tokens[0])
    text = llm.tokenizer.decode(answer_tokens, skip_special_tokens=True)
    print(text[len(shown[0]):], end="", flush=True)
    shown[0] = text

result = llm.generate("Why does a chatbot show its first word later than the rest? Answer in three sentences.",
                      max_new_tokens=160, on_token=show)
print("\n\n", result)

Two numbers came back with the text: the time to first token and the time per decode step. Next: how does each one change when the prompt gets longer?

## 2. Both phases against prompt length

The helper below builds prompts of a chosen length by repeating a paragraph. For each length, generate 32 tokens and record the time to first token and the milliseconds per decode step.

**Fill in `measure`.** Call `llm.generate` with `stop_at_eos=False`, so every run produces exactly `new_tokens` tokens and the runs are comparable. Return the actual prompt length, the TTFT and the milliseconds per token from the result.

In [ ]:
PARAGRAPH = ("The workshop room is quiet except for keyboards. Under each laptop a chip reads the same "
             "weights again and again, one token at a time. ")

def prompt_of_length(n_tokens):
    text = PARAGRAPH
    while len(llm.tokenizer.encode(text)) < n_tokens:
        text += PARAGRAPH
    return text + " Summarise the above in one sentence."

def measure(n_tokens, new_tokens=config.ACT1_NEW_TOKENS):
    """Returns (prompt_length, ttft_ms, decode_ms_per_token) for a prompt of about n_tokens tokens."""
    # >>> SOLUTION BEGIN: call llm.generate once and print the result to see what it reports; help(llm.generate) lists the arguments
    r = llm.generate(prompt_of_length(n_tokens), max_new_tokens=new_tokens, stop_at_eos=False)
    return r.prompt_lens[0], r.ttft_ms, r.decode_ms_per_token
    # <<< SOLUTION END

rows = [measure(n) for n in config.ACT1_PROMPT_LENGTHS]
for length, ttft, ms in rows:
    print(f"prompt {length:>5} tokens   TTFT {ttft:7.0f} ms   decode {ms:6.1f} ms/token ({1000 / ms:5.1f} tokens/s)")

In [ ]:
lengths, ttfts, mss = zip(*rows)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(lengths, ttfts, "o-"); ax[0].set(xlabel="prompt length (tokens)", ylabel="time to first token (ms)", title="Prefill")
for step in config.PREFILL_STEPS:                  # prompts are padded up to these lengths (see below)
    ax[0].axvline(step, ls=":", c="gray")
ax[1].plot(lengths, [1000 / m for m in mss], "s-"); ax[1].set(xlabel="prompt length (tokens)", ylabel="tokens per second", title="Decode")
ax[1].set_ylim(bottom=0)
for a in ax: a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

The time to first token grows with the prompt. Decode speed hardly changes.

Look closely at the prefill curve: it climbs in steps. tt_transformers rounds every prompt up to 128, 1,024, 2,048 or 4,096 tokens (the dotted lines) before processing it, so it can reuse a few prepared programs instead of preparing one for every length. Prompts inside one step cost about the same: a 1,200-token prompt takes as long as a 1,950-token one. That is why three of the lengths above fall into each step.

Now compare how many tokens per second the chip handles in each phase:

In [ ]:
length, ttft, ms = rows[-1]
prefill_tps = length / (ttft / 1000)
decode_tps = 1000 / ms
print(f"prefill: {length} prompt tokens in {ttft:.0f} ms  ->  {prefill_tps:8.0f} tokens per second")
print(f"decode:  one token every {ms:.1f} ms             ->  {decode_tps:8.0f} tokens per second")
print(f"same model, same card: prefill handles {prefill_tps / decode_tps:.0f}x more tokens per second")

## 3. Why: compute-bound and memory-bound

Two terms first:

- **Compute-bound:** the time is set by how much arithmetic the cores have to do.
- **Memory-bound:** the time is set by how many bytes have to travel from memory to the cores.

Before the chip can compute with a weight, the weight has to travel from the card's main memory (**DRAM**, the 12 GB) to the cores.

- In **decode**, each weight that arrives is used once, for one multiplication and one addition, and then the next weight is needed. The arithmetic is quick; waiting for the next weight is not. Decode is **memory-bound**: its speed is set by how many bytes per second memory can deliver (the **memory bandwidth**).
- In **prefill**, each weight that arrives is used for every token of the prompt, so hundreds of operations happen per byte read. Now the arithmetic takes longer than the reading. Prefill is **compute-bound**.

That is why prefill gets through so many more tokens per second: the weights are read once for the whole prompt instead of once per token.

## 4. The speed limit

If decode is memory-bound, a speed limit follows from two numbers:

$$\text{minimum time per token} = \frac{\text{bytes read per decode step}}{\text{memory bandwidth}}$$

One decode step reads every weight of every layer once, plus the output layer. The embedding table is the exception: only one row of it is read. `llm.facts` adds this up using the number format each group of weights actually has on the card.

The limit belongs to the model and the memory, not to a particular chip: the same formula works for any accelerator, GPUs included. It holds for one user, with these weights in these number formats. Smaller formats lower the bytes (Act III); batching and speculative decoding read the weights once for several tokens (Act III, Talk 3).

**Fill in `speed_limit`.**

In [ ]:
def speed_limit(bytes_per_token, bandwidth_gbs):
    """Returns (minimum ms per token, maximum tokens per second) at batch size 1."""
    # >>> SOLUTION BEGIN: the formula is in the text above; watch the units: 1 GB/s is 1e9 bytes per second
    seconds = bytes_per_token / (bandwidth_gbs * 1e9)
    return seconds * 1e3, 1 / seconds
    # <<< SOLUTION END

facts = llm.facts
min_ms, max_tps = speed_limit(facts["weight_bytes_per_token"], llm.card["bandwidth_gbs"])
measured_ms = rows[0][2]
print(f"bytes read per decode step: {facts['weight_bytes_per_token'] / 1e9:.2f} GB")
print(f"speed limit on this card:   {min_ms:.1f} ms per token = {max_tps:.0f} tokens/s")
print(f"measured:                   {measured_ms:.1f} ms per token = {1000 / measured_ms:.0f} tokens/s "
      f"({100 * min_ms / measured_ms:.0f}% of the limit)")

The measurement is well above the limit. Two things fill the gap:

1. **The host sending commands (dispatch).** Without trace, the host sends every command of every decode step itself. On the n150 you are using, that makes one step take about 65 ms instead of 22 ms (we measured it with `tools/check_dispatch.py`). Trace removes nearly all of it, which is why it stays on.
2. **Time on the card itself.** One decode step of this model runs several hundred separate operations, roughly two dozen per layer across 28 layers. Each has a small fixed cost to start and finish, however little data it moves, and not every weight streams at the full bandwidth. For a model this small, those costs add up to more than the time spent reading the weights. Larger models get much closer to the limit, because their weight reads dominate. It is also why production code fuses several operations into one. Splitting this time up further needs the profiler, which is beyond today.

So the speed limit is a ceiling for this setup (one user, these number formats), not a prediction. Decode is still the phase that is starved of work: Act III shows how to give the card more to do in each step.

**Free the card for the next act.** A program that has used the card keeps it until it stops, so the last cell stops this notebook's kernel. Your outputs stay on the page; VS Code will show the kernel as stopped, which is expected.

The same cell starts warm-up part 2, which prepares Acts II to IV: about 9 minutes the first time on a machine, about 1.5 minutes once compiled. Its progress goes to `notebooks/warmup2.log`; Act II waits for it if needed.

In [ ]:
llm.close()
from common import start_warmup, release_card
start_warmup(2)   # prepares Acts II-IV in the background, during the break
print("Act I done.")
release_card()